# Problem 1

In [1]:
Text1 = """To qualify for the advanced training program, you need prior experience in the role, but the
only way to gain that experience is by completing the advanced training program."""

Text2 = """You can only access the advanced training if you've done the job before, but you can't do
the job without the training."""


### i. Sentence Tokenizer

In [2]:
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import MultiLabelBinarizer

tokens1, tokens2 = word_tokenize(Text1), word_tokenize(Text2)

tokens1_clean = [w.lower() for w in tokens1 if w.isalpha()]
tokens2_clean = [w.lower() for w in tokens2 if w.isalpha()]

mlb = MultiLabelBinarizer()
X = mlb.fit_transform([tokens1, tokens2])
X_clean = mlb.fit_transform([tokens1_clean, tokens2_clean])

In [3]:
print(tokens1)
print(tokens2)
print(tokens1_clean)
print(tokens2_clean)
print(X[0])
print(X[1])
print(X_clean[0])
print(X_clean[1])


['To', 'qualify', 'for', 'the', 'advanced', 'training', 'program', ',', 'you', 'need', 'prior', 'experience', 'in', 'the', 'role', ',', 'but', 'the', 'only', 'way', 'to', 'gain', 'that', 'experience', 'is', 'by', 'completing', 'the', 'advanced', 'training', 'program', '.']
['You', 'can', 'only', 'access', 'the', 'advanced', 'training', 'if', 'you', "'ve", 'done', 'the', 'job', 'before', ',', 'but', 'you', 'ca', "n't", 'do', 'the', 'job', 'without', 'the', 'training', '.']
['to', 'qualify', 'for', 'the', 'advanced', 'training', 'program', 'you', 'need', 'prior', 'experience', 'in', 'the', 'role', 'but', 'the', 'only', 'way', 'to', 'gain', 'that', 'experience', 'is', 'by', 'completing', 'the', 'advanced', 'training', 'program']
['you', 'can', 'only', 'access', 'the', 'advanced', 'training', 'if', 'you', 'done', 'the', 'job', 'before', 'but', 'you', 'ca', 'do', 'the', 'job', 'without', 'the', 'training']
[0 1 1 1 0 0 1 0 1 1 0 0 1 0 0 1 1 1 0 1 1 0 0 1 1 1 1 1 1 1 1 1 1 1 0 1]
[1 1 1 0 1 

In [4]:
import numpy as np

def sim_costeta(_a:np.ndarray, _b:np.ndarray)->float:
    from numpy.linalg import norm
    assert _a.shape == _b.shape, 'require vectors same shape'
    return np.dot(_a, _b) / (norm(_a)*norm(_b))

def sim_jaccard(_a:np.ndarray, _b:np.ndarray)->float:
    ix = np.where((_a>0.) & (_b>0.))
    ixd = np.where((_a>0.) | (_b>0.))
    return ix[0].shape[0] / ixd[0].shape[0]

In [5]:
cos_sim = sim_costeta(X[0], X[1])
jac_sim = sim_jaccard(X[0], X[1])

print("Cosine similarity:", cos_sim)
print("Jaccard similarity:", jac_sim)

Cosine similarity: 0.3651483716701107
Jaccard similarity: 0.2222222222222222


In [6]:
cos_sim_clean = sim_costeta(X_clean[0], X_clean[1])
jac_sim_clean = sim_jaccard(X_clean[0], X_clean[1])

print("Cosine similarity:", cos_sim_clean)
print("Jaccard similarity:", jac_sim_clean)

Cosine similarity: 0.3380617018914066
Jaccard similarity: 0.2


The similarity is relatively low because the tokenizer + bag-of-words approach only considers exact word overlap. Both sentences express the same circular dependency, but they use different vocabulary, so the model fails to capture their semantic similarity. For example, the model considers:

- "experience" and "done the job" as unrelated
- "qualify" and "access" as unrelated.

We notice the Cosine similarity is higher than the Jaccard similarity. Cosine is vector based: $\frac{A \cdot B}{||A||||B||}$, looking at the angle between the two vectors. As such, penalizes non-overlapping words less than Jaccard which is set based, $\frac{A \cap B}{A \cup B}$, only looking at which words are shared.

The similarity scores are slightly lower for the "clean" version which removes punctuation and case differences. Removing the case differences make the sentences more similar, but that's overshadowed by removing punctuation which inflates the similarity, i.e., there are fewer case differences than there is punctuation in the two sentences.

### ii. TFIDF

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(lowercase=True, stop_words=None)

X_tfidf = vec.fit_transform([Text1, Text2])

print("TF-IDF matrix shape:", X_tfidf.shape)
print("Vocabulary:", vec.get_feature_names_out())

tfidf_cos = sim_costeta(X_tfidf.toarray()[0], X_tfidf.toarray()[1])
print("TF-IDF cosine similarity:", tfidf_cos)

tfidf_jaccard = sim_jaccard(X_tfidf.toarray()[0], X_tfidf.toarray()[1])
print("TF-IDF jaccard similarity:", tfidf_jaccard)

TF-IDF matrix shape: (2, 30)
Vocabulary: ['access' 'advanced' 'before' 'but' 'by' 'can' 'completing' 'do' 'done'
 'experience' 'for' 'gain' 'if' 'in' 'is' 'job' 'need' 'only' 'prior'
 'program' 'qualify' 'role' 'that' 'the' 'to' 'training' 've' 'way'
 'without' 'you']
TF-IDF cosine similarity: 0.3987108072273213
TF-IDF jaccard similarity: 0.2


We see a slight increase in cosine-similarity with TF-IDF. This is because TF-IDF improves on plain token overlap by down-weighting very common words and emphasizing more informative terms. However, it still depends on shared vocabulary and does not capture paraphrases as well as embedding-based methods.

### iii. Word Embeddings

In [8]:
import gensim
from gensim.models import Word2Vec
from nltk.corpus import abc

sentences = list(abc.sents())

w2v = Word2Vec(
    sentences=sentences,
    min_count=2,
    workers=4,
    sg=1      # skip-gram; set sg=0 for CBOW
)

def sentence_embedding(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

s1_vec = sentence_embedding(tokens1, w2v)
s2_vec = sentence_embedding(tokens2, w2v)

emb_cos = sim_costeta(s1_vec, s2_vec)
print("Word embedding cosine similarity:", emb_cos)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word embedding cosine similarity: 0.96285945


The cosine similarity is high because averaging word embeddings removes sentence structure and emphasizes shared vocabulary and semantic similarity, causing different sentences with similar meanings to appear nearly identical in vector space. Word2Vec produces word-level embeddings, not sentence-level ones. When we average all word vectors in a sentence, we lose:

- word order
- syntax
- negation and logical structure

Both sentences use many similar or related words (e.g., training, program, experience, job),so we end up with very similar averaged vectors.

Note: Jaccard similarity is not appropriate here because word embeddings are continuous-valued vectors rather than binary.

### iv. Sentence Transformer

In [9]:
import torch
import transformers
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# filter warnings
import warnings
transformers.logging.set_verbosity_error()
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

Device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f'PyTorch version= {torch.__version__}')
print(f'transformers version= {transformers.__version__}')
print(f'CUDA available= {torch.cuda.is_available()}')

PyTorch version= 2.10.0
transformers version= 5.0.0
CUDA available= False


In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

s1_vec = model.encode(Text1)
s2_vec = model.encode(Text2)

sim = sim_costeta(s1_vec, s2_vec)

print("Sentence transformer cosine similarity:", sim)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sentence transformer cosine similarity: 0.68473685


We see the Sentence Transformer model produces a cosine similarity of approximately 0.68, which is significantly lower than the Word2Vec result (0.96), and more likely realistic.

Sentence Transformers generate context-aware sentence-level embeddings. This means the model considers:

- word order
- sentence structure
- relationships between words
- overall meaning of the sentence

As a result, it captures the fact that the two sentences express a similar idea (a circular dependency between experience and training), but are not identical in wording or structure.

The similarity score tells us the sentences are clearly related, but they use different phrasing and structure, i.e., their similar, but not identical. 

### v. BERT

BERT generates a sequence of token-level embeddings, each capturing the context of the entire input. We need a single fixed-length vector, so the code below uses mean pooling. Mean pooling averages all token embeddings, ensuring that every token contributes equally to the final representation as described here: https://milvus.io/ai-quick-reference/why-is-mean-pooling-often-used-on-the-token-outputs-of-a-transformer-like-bert-to-produce-a-sentence-embedding

In [11]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

model_name = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

sentences = [Text1, Text2]

encoded = tokenizer(
    sentences,
    padding=True,
    truncation=True,
    return_tensors="pt"
)

with torch.no_grad():
    output = model(**encoded)

sentence_embeddings = mean_pooling(output, encoded["attention_mask"])
sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

bert_cos = torch.matmul(sentence_embeddings[0], sentence_embeddings[1]).item()
print("BERT cosine similarity:", bert_cos)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BERT cosine similarity: 0.8650065064430237


Unlike Word2Vec, BERT captures context, word order, and sentence structure, allowing it to better represent the meaning of each sentence.

However, BERT is not specifically trained for sentence similarity tasks, so mean pooling is uses as a heuristic for creating sentence embeddings. The similarity score is likely more realistic than Word2Vec, but typically probably less reliable than Sentence Transformers, which are explicitly trained on sentence pairs.

Sentence Transformers is purposefully built for the task of comparing sentences and is the most reliable:
- It is trained specifically to capture the overall meaning of a sentence, rather than just looking at word overlaps. 
- It creates embeddings that can be compared instantly with cosine similarity, without having do something like mean pooling like we did with BERT.
- It handles paraphrased sentences, and understands that different words can share the same meaning (e.g., "how to reset a password" vs. "recover account credentials").

https://huggingface.co/blog/how-to-train-sentence-transformers

# Problem 2

The topic retrieval in step 1 could sometimes reduce accuracy by restricting the search space too early. If the wrong topic is selected, the correct paragraph may never be considered. In Cell 9, where the model selects a general topic (Geography of the United States), instead of a more relevant one (Alaska).

However, Step 1 also improves efficiency by narrowing the search space, which is useful for large datasets. Without it, similarity would need to be computed across all paragraphs which would be more expensive and impractical for very large datasets. In my testing, the search completed in about 12 seconds with topic retrieval, and took over a minute without topic retrieval, so more than a 5x slow down.

The trade-off is between efficiency and accuracy. I would not skip step 1 all together, but I might take a hybrid approach and use the top-k topics. The code below demonstrates this. When I tried it, the topics returned were: 

Top-5 topics: ['Geography_of_the_United_States', '51st_state', 'Alaska', 'Montana', 'Florida']

The paragraph returned was about Montana, mentioning Alaska as the largest state, and the search time was just over 12 seconds.

In [12]:
from sklearn.metrics.pairwise import cosine_similarity

# Modified class that uses top-k topics in the generate_answer method.
class RAGLLM:
    def __init__(self, external_data):
        self.st_model = SentenceTransformer('all-MiniLM-L6-v2')  # sentence transformer is used for embeddings
        
        # external data
        self.external_data = external_data
        self.titles = self.external_data['topic'].unique()
        self.title_embeddings = self.st_model.encode(self.titles.tolist())  # embed each topic/title

        # LLM model
        self.generator = pipeline('text-generation', model='meta-llama/Llama-3.2-1B-Instruct',
                                  max_length=4096, device=Device)  # utilize GPU in this example

    def generate_answer(self, user_question):
        # Step 1: Find the top-k best matching topic
        question_embedding = self.st_model.encode([user_question])
        title_similarities = cosine_similarity(question_embedding, self.title_embeddings)[0]

        k = 5
        top_k_indices = np.argsort(title_similarities)[-k:][::-1]
        top_k_titles = [str(self.titles[i]) for i in top_k_indices]

        print(f"Top-{k} topics: {top_k_titles}\n")

        # # Step 2: Filter contexts for the top-k best topics
        topic_contexts = self.external_data[self.external_data['topic'].isin(top_k_titles)]['context'].tolist()

        # Step 3: Vectorize each context under the best topic and find the best matching context
        context_embeddings = self.st_model.encode(topic_contexts)
        context_similarities = cosine_similarity(question_embedding, context_embeddings)
        best_context_index = np.argmax(context_similarities)
        best_context = topic_contexts[best_context_index]
        print(f"The best paragraph: {best_context}\n")

        # Step 4: Generate the answer using LLM with retrieved context
        prompt = f"Context: {best_context}\n\nQuestion: {user_question}\nAnswer:"
        response = self.generator(prompt, max_length=300)
        answer_text = response[0]['generated_text']
        answer = answer_text.split("Answer: ")[-1] 
        return answer

# Problem 3


The output will likely be the same because the external data set (SQuAD v2.0) only contains information up to around 2018, and the retrieved context specifically includes population data from 2015. Even though the LLM itself may have been trained on more recent data, in the RAG system it should prioritize the provided context when generating an answer. As a result, the output  should reflect the outdated external dataset rather than the actual 2022 population.

I went ahead and tested this using the notebook:

Here is the output for "What is the population of Florida?":

`20,271,272 as estimated by the United States Census Bureau on July 1, 2015.
Note: The population of Florida was 18,801,310 in the 2010 census.`

Here is the output for "What is the population of Florida in 2025?":

`The population of Florida is estimated to be 21,357,841 by the United States Census Bureau, based on the 2020 United States Census. This estimate is higher than the 2020 United States Census population of 21,216,784. The United States Census Bureau projects that the population of Florida will continue to grow, but at a slower rate than in the 2010 census.`

In this case, the model did return more recent data. This likely occurs because the LLM is not strictly constrained to use the retrieved context, given our prompt setup. That allows for the LLM to use the retrieved information with its own pre-trained knowledge, which appears to include more recent data. Since the query explicitly references a future year (2025), the model must have recognized the retrieved context was outdated and overrode it with a more plausible estimate.

If we made our prompt stricter, e.g., "Answer ONLY using the provided context. If the answer is not in the context, say Not found." than the model will at least let us know if it is in the context or not. I tried it and got this response:

`Not found Context: The population of Florida in 2025 is unknown because the United States Census Bureau does not release population estimates for years after the 2020 census. The last population estimate for Florida was released in 2020, and it was 21,787,169. However, the Census Bureau will release population estimates for 2025`

We see it did say "not found," but then preceded to answers with its more recent data regardless of our prompt. This is interesting as it demonstrates that instructions alone are insufficient to guarantee grounding in RAG systems.

# Problem 4

The model produces the correct answer even though the retrieved topic (Appalachian Mountains) and context (mountains in Pennsylvania) are irrelevant because the LLM relies not only on the retrieved context, but also on its pre-trained knowledge. In this case, the question (“Where is Mount Rushmore?”) is simple and widely known, so the LLM likely already knows the answer from training data.

This means the model is not truly using the retrieved context but instead is relying on its internal knowledge which means the answer is not grounded in just the context, as we saw in problem 3 above. 



# Problem 5

RAG-LLM requires access to additional data source(s), and finding relevant context within the data source(s) may be time consuming. Performance for RAG depends heavily on the quality and relevance of the retrieved data. If the retriever selects incorrect or irrelevant context, the final answer may be inaccurate. It is also more complex to implement and maintain due to the need for both retrieval and generation components.

In contrast, a general LLM may be simpler to use and performs well on tasks involving general knowledge, reasoning, or common questions. It does not rely on an external dataset and can generate fluent, coherent responses quickly. However, its knowledge is limited to its training data, which may be outdated, and it is more prone to hallucination since it is not grounded in external evidence.

In practice, RAG-LLM should be used when accuracy, freshness, and domain specificity are important, while a general LLM is preferable for broad, general-purpose tasks where simplicity and speed are more important.

# Problem 6

In [13]:
import json
import pandas as pd

# explore SQuAD training data structure
with open('EP_datasets/squad-train-v2.0.json', 'r') as f:
    squad_data = json.load(f)

n_titles = len(squad_data['data'])
print(f'There are {n_titles} titles/topics in SQuAD v2.0 training dataset.\n')

title_idx = 200
prg_idx = 2
current_title = squad_data['data'][title_idx]['title']
n_paragraphs = len(squad_data['data'][title_idx]['paragraphs'])
n_qas = len(squad_data['data'][title_idx]['paragraphs'][prg_idx]['qas'])
print(f"Title with index #{title_idx} is {squad_data['data'][title_idx]['title']}.")
print(f"Title {current_title} has {n_paragraphs} paragraphs. Paragraph with index {prg_idx} has {n_qas} question-answer pairs.")

There are 442 titles/topics in SQuAD v2.0 training dataset.

Title with index #200 is Florida.
Title Florida has 35 paragraphs. Paragraph with index 2 has 10 question-answer pairs.


In [14]:
sqdata = {"topic": [], "context": []}

for i in range(n_titles):
    n_paragraphs = len(squad_data['data'][i]['paragraphs'])
    for j in range(n_paragraphs):
        sqdata["topic"].append(squad_data['data'][i]['title'])
        sqdata["context"].append(squad_data['data'][i]['paragraphs'][j]['context'])
    

df = pd.DataFrame(sqdata)

# sanity
print(f'Shape of df: {df.shape}')
df.head()

Shape of df: (19035, 2)


,topic,context
0,Beyoncé,Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ b...
1,Beyoncé,Following the disbandment of Destiny's Child i...
2,Beyoncé,"A self-described ""modern-day feminist"", Beyonc..."
3,Beyoncé,"Beyoncé Giselle Knowles was born in Houston, T..."
4,Beyoncé,Beyoncé attended St. Mary's Elementary School ...


We add a method to generate an answer with no context to make comparison easy.

In [17]:
class RAGLLM:
    def __init__(self, external_data):
        self.st_model = SentenceTransformer('all-MiniLM-L6-v2')  # sentence transformer is used for embeddings
        
        # external data
        self.external_data = external_data
        self.titles = self.external_data['topic'].unique()
        self.title_embeddings = self.st_model.encode(self.titles.tolist())  # embed each topic/title

        # LLM model
        self.generator = pipeline('text-generation', model='meta-llama/Llama-3.2-1B-Instruct',
                                  max_length=4096, device=Device)  # utilize GPU in this example

    def generate_answer(self, user_question):
        # Step 1: Find the top-k best matching topic
        question_embedding = self.st_model.encode([user_question])
        title_similarities = cosine_similarity(question_embedding, self.title_embeddings)[0]

        k = 5
        top_k_indices = np.argsort(title_similarities)[-k:][::-1]
        top_k_titles = [str(self.titles[i]) for i in top_k_indices]

        print(f"Top-{k} topics: {top_k_titles}\n")

        # # Step 2: Filter contexts for the top-k best topics
        topic_contexts = self.external_data[self.external_data['topic'].isin(top_k_titles)]['context'].tolist()

        # Step 3: Vectorize each context under the best topic and find the best matching context
        context_embeddings = self.st_model.encode(topic_contexts)
        context_similarities = cosine_similarity(question_embedding, context_embeddings)
        best_context_index = np.argmax(context_similarities)
        best_context = topic_contexts[best_context_index]
        print(f"The best paragraph: {best_context}\n")

        # Step 4: Generate the answer using LLM with retrieved context
        prompt = f"Context: {best_context}\n\nQuestion: {user_question}\nAnswer:"
        response = self.generator(prompt, max_length=300)
        answer_text = response[0]['generated_text']
        answer = answer_text.split("Answer: ")[-1] 
        return answer
    
    def generate_answer_with_no_context(self, user_question):
        # Step 4: Generate the answer using LLM with retrieved context
        prompt = f"Question: {user_question}\nAnswer:"
        response = self.generator(prompt, max_length=300)
        answer_text = response[0]['generated_text']
        answer = answer_text.split("Answer: ")[-1] 
        return answer

The SQUAD data Beyoncé topic contains a paragraph with the phrase "The Daily Mail calls Beyoncé's voice "versatile", capable of exploring power ballads, soul, rock belting, operatic flourishes, and hip hop." 

We'll ask the LLM how the Daily Mail describes Beyoncé's voice with and without context to see what happens.


In [19]:
query = "What does The Daily Mail say about Beyoncé's voice?"
ragllm = RAGLLM(df)
output = ragllm.generate_answer(query)
for part in output.split("\n\n"):
    print("With context: ", part)

output = ragllm.generate_answer_with_no_context(query)
for part in output.split("\n\n"):
    print("Without context: ", part)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Top-5 topics: ['Beyoncé', 'Madonna_(entertainer)', 'Dialect', 'Queen_(band)', 'Phonology']

The best paragraph: Beyoncé's vocal range spans four octaves. Jody Rosen highlights her tone and timbre as particularly distinctive, describing her voice as "one of the most compelling instruments in popular music". While another critic says she is a "Vocal acrobat, being able to sing long and complex melismas and vocal runs effortlessly, and in key. Her vocal abilities mean she is identified as the centerpiece of Destiny's Child. The Daily Mail calls Beyoncé's voice "versatile", capable of exploring power ballads, soul, rock belting, operatic flourishes, and hip hop. Jon Pareles of The New York Times commented that her voice is "velvety yet tart, with an insistent flutter and reserves of soul belting". Rosen notes that the hip hop era highly influenced Beyoncé's strange rhythmic vocal style, but also finds her quite traditionalist in her use of balladry, gospel and falsetto. Other critics prais

We see that, with RAG, we get the detailed and expected result, ""versatile", capable of exploring power ballads, soul, rock belting, operatic flourishes, and hip hop."

Without RAG, the reply states "I couldn't find any specific article from The Daily Mail" and proceeds to give a generic answer. The model fails to answer the question and does not retrieve the specific description (“versatile”), demonstrating lack of access to the required information.

Let's try one more that is an explicit fact, "In what city did Beyoncé attend elementary school?"

In [21]:
query = "In what city did Beyoncé attend elementary school?"
ragllm = RAGLLM(df)
output = ragllm.generate_answer(query)
for part in output.split("\n\n"):
    print("With context: ", part)

output = ragllm.generate_answer_with_no_context(query)
for part in output.split("\n\n"):
    print("Without context: ", part)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Top-5 topics: ['Beyoncé', 'Comprehensive_school', 'Eton_College', 'Washington_University_in_St._Louis', 'University']

The best paragraph: Beyoncé attended St. Mary's Elementary School in Fredericksburg, Texas, where she enrolled in dance classes. Her singing talent was discovered when dance instructor Darlette Johnson began humming a song and she finished it, able to hit the high-pitched notes. Beyoncé's interest in music and performing continued after winning a school talent show at age seven, singing John Lennon's "Imagine" to beat 15/16-year-olds. In fall of 1990, Beyoncé enrolled in Parker Elementary School, a music magnet school in Houston, where she would perform with the school's choir. She also attended the High School for the Performing and Visual Arts and later Alief Elsik High School. Beyoncé was also a member of the choir at St. John's United Methodist Church as a soloist for two years.

With context:  Fredericksburg, Texas.
With context:  Beyoncé's story is one of perseve

With RAG we get the correct answer "Fredericksburg, Texas." Without RAG, we get an incorrect answer, "Houston, Texas."